# C3 — Colectare comentarii YouTube
În acest notebook colectăm un eșantion  de comentarii publice de pe YouTube.
Scopul nu este să obținem corpusul final mare, ci să înțelegem fluxul:
sursă → API → comentarii brute → fișier JSONL.
La final, fiecare student salvează propriul fișier în `data/raw/`.

## 1. Ce trebuie să avem pregătit
Avem nevoie de:
- fișier `.env` în root-ul proiectului
- cheia `YOUTUBE_API_KEY`
- un handle de canal YouTube
Exemplu în `.env`:
```text
YOUTUBE_API_KEY=cheia_ta_aici

In [26]:

from pathlib import Path
import os
import json
import requests
from datetime import datetime
from dotenv import load_dotenv

## 2. Încărcăm cheia API
Notebook-ul caută fișierul `.env` în root-ul proiectului.
Dacă cheia nu este găsită, colectarea nu poate porni.

In [27]:
ROOT = Path.cwd()
while not (ROOT / ".env").exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
load_dotenv(ROOT / ".env")
API_KEY = os.getenv("YOUTUBE_API_KEY")
BASE_URL = "https://www.googleapis.com/youtube/v3"
print("Root proiect:", ROOT)
print("Cheie găsită:", API_KEY is not None)

Root proiect: c:\Users\iarin\Desktop\inginerie AI\echochamber-project-team-6
Cheie găsită: True


## 3. Alegem canalul și numărul de videoclipuri
Fiecare student schimbă `student_id` și `handle`.
Pentru exercițiu folosim puține videoclipuri, ca să nu consumăm inutil cota API.

In [28]:
student_id = "student_01"
handle = "VictorPonta1"
max_videos = 10
max_comments_per_video = 10
output_file = ROOT / "data" / "raw" / f"{student_id}_youtube_raw.jsonl"
print(output_file)

c:\Users\iarin\Desktop\inginerie AI\echochamber-project-team-6\data\raw\student_01_youtube_raw.jsonl


## 4. Găsim canalul YouTube

YouTube lucrează intern cu `channel_id`, nu direct cu numele canalului.
De aceea, primul pas este să transformăm handle-ul în `channel_id`.

In [29]:
channel_response = requests.get(
    f"{BASE_URL}/channels",
    params={
        "part": "id",
        "forHandle": handle,
        "key": API_KEY
    }
)
channel_data = channel_response.json()
channel_data

{'kind': 'youtube#channelListResponse',
 'etag': 'x0BqextaubB_iy38gSG9vfNaiEY',
 'pageInfo': {'totalResults': 1, 'resultsPerPage': 5},
 'items': [{'kind': 'youtube#channel',
   'etag': 'Bm01WdsOMwehiBdckiS17kz1wnk',
   'id': 'UCBBebBcNsrTgSDZWLXltK0g'}]}

In [30]:
channel_id = channel_data["items"][0]["id"]
channel_id

'UCBBebBcNsrTgSDZWLXltK0g'

## 5. Luăm cele mai recente videoclipuri
Acum cerem ultimele videoclipuri publicate de canal.
Pentru curs folosim doar câteva videoclipuri.

In [31]:
videos_response = requests.get(
    f"{BASE_URL}/search",
    params={
        "part": "snippet",
        "channelId": channel_id,
        "type": "video",
        "order": "date",
        "maxResults": max_videos,
        "key": API_KEY
    }
)
videos_data = videos_response.json()
videos_data["items"][0]

{'kind': 'youtube#searchResult',
 'etag': '0qUzxn4biGjtJvImg5o-Gu5D4VU',
 'id': {'kind': 'youtube#video', 'videoId': 'xRvIwmt0uag'},
 'snippet': {'publishedAt': '2026-05-11T14:59:00Z',
  'channelId': 'UCBBebBcNsrTgSDZWLXltK0g',
  'title': 'Programul SAFE: Afacerea Olandeză și Contractul cu Nemții #shorts',
  'description': 'Controverse privind programul SAFE: Investiții sau înstrăinare de active strategice? Bani de la stat pentru firme olandeze și ...',
  'thumbnails': {'default': {'url': 'https://i.ytimg.com/vi/xRvIwmt0uag/default.jpg',
    'width': 120,
    'height': 90},
   'medium': {'url': 'https://i.ytimg.com/vi/xRvIwmt0uag/mqdefault.jpg',
    'width': 320,
    'height': 180},
   'high': {'url': 'https://i.ytimg.com/vi/xRvIwmt0uag/hqdefault.jpg',
    'width': 480,
    'height': 360}},
  'channelTitle': 'Victor Ponta',
  'liveBroadcastContent': 'none',
  'publishTime': '2026-05-11T14:59:00Z'}}

In [32]:
videos = []
for item in videos_data["items"]:
    videos.append({
        "video_id": item["id"]["videoId"],
        "video_title": item["snippet"]["title"],
        "video_date": item["snippet"]["publishedAt"][:10]
    })
videos

[{'video_id': 'xRvIwmt0uag',
  'video_title': 'Programul SAFE: Afacerea Olandeză și Contractul cu Nemții #shorts',
  'video_date': '2026-05-11'},
 {'video_id': 'OCI4GZ8_XiU',
  'video_title': 'Creator Digital vs. Românii: Banii din Dubai și Lecțiile #shorts',
  'video_date': '2026-05-11'},
 {'video_id': 'GncqjECzhOw',
  'video_title': 'Președintele României: Adevărul despre Kövesi și Guvernare #shorts',
  'video_date': '2026-05-11'},
 {'video_id': 'CpLZHS-6LtM',
  'video_title': 'RomaniaTV live 10 mai 2026',
  'video_date': '2026-05-10'},
 {'video_id': 'GrP9oTni12s',
  'video_title': 'Live May 10',
  'video_date': '2026-05-10'},
 {'video_id': 'yJiXoWUIOto',
  'video_title': 'Guvern Stabil pentru România: Prioritate Economică și Socială #shorts',
  'video_date': '2026-05-07'},
 {'video_id': 'nvaahazQC0c',
  'video_title': 'Priorități România: Guvernul pentru Cetățeni, Nu FMI! #shorts',
  'video_date': '2026-05-06'},
 {'video_id': 'j2hHOhYaOVM',
  'video_title': 'Problemele Reale ale Rom

## 6. Colectăm comentariile
Pentru fiecare videoclip luăm comentariile publice ordonate după relevanță.
În acest exercițiu nu folosim paginare, deci luăm maximum 100 comentarii per videoclip.

In [33]:
comments = []
for video in videos:
    print("Colectez:", video["video_title"][:80])
    comments_response = requests.get(
        f"{BASE_URL}/commentThreads",
        params={
            "part": "snippet",
            "videoId": video["video_id"],
            "maxResults": max_comments_per_video,
            "textFormat": "plainText",
            "order": "relevance",
            "key": API_KEY
        }
    )
    comments_data = comments_response.json()
    for comment_item in comments_data.get("items", []):
        snippet = comment_item["snippet"]["topLevelComment"]["snippet"]
        record = {
            "id": f"yt_{video['video_id']}_{comment_item['id']}",
            "source_platform": "youtube",
            "source_channel": handle,
            "text_raw": snippet["textDisplay"],
            "video_id": video["video_id"],
            "video_title": video["video_title"],
            "video_date": video["video_date"],
            "comment_date": snippet["publishedAt"][:10],
            "likes": snippet["likeCount"],
            "collected_at": datetime.utcnow().strftime("%Y-%m-%d")
        }
        comments.append(record)
len(comments)

Colectez: Programul SAFE: Afacerea Olandeză și Contractul cu Nemții #shorts
Colectez: Creator Digital vs. Românii: Banii din Dubai și Lecțiile #shorts


C:\Users\iarin\AppData\Local\Temp\ipykernel_2732\3206550858.py:28: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  "collected_at": datetime.utcnow().strftime("%Y-%m-%d")


Colectez: Președintele României: Adevărul despre Kövesi și Guvernare #shorts
Colectez: RomaniaTV live 10 mai 2026
Colectez: Live May 10
Colectez: Guvern Stabil pentru România: Prioritate Economică și Socială #shorts
Colectez: Priorități România: Guvernul pentru Cetățeni, Nu FMI! #shorts
Colectez: Problemele Reale ale României: Inflație, Educație, Sănătate #shorts
Colectez: Criza Guvernamentală: PSD, PNL și Viitorul Politic #shorts
Colectez: Live May 05


48

# Explorare si curatare

## 7. Inspectăm primele comentarii
Înainte să salvăm fișierul, verificăm dacă datele arată cum trebuie.

In [34]:
comments[:3]

[{'id': 'yt_OCI4GZ8_XiU_UgwksPU95OgaL6b3SFp4AaABAg',
  'source_platform': 'youtube',
  'source_channel': 'VictorPonta1',
  'text_raw': 'Super tare domnul Ponta ❤❤❤',
  'video_id': 'OCI4GZ8_XiU',
  'video_title': 'Creator Digital vs. Românii: Banii din Dubai și Lecțiile #shorts',
  'video_date': '2026-05-11',
  'comment_date': '2026-05-11',
  'likes': 0,
  'collected_at': '2026-05-11'},
 {'id': 'yt_OCI4GZ8_XiU_UgwFVgDUjuk8iaJg82h4AaABAg',
  'source_platform': 'youtube',
  'source_channel': 'VictorPonta1',
  'text_raw': 'Vocabular de bisnitar la piața mare!!!',
  'video_id': 'OCI4GZ8_XiU',
  'video_title': 'Creator Digital vs. Românii: Banii din Dubai și Lecțiile #shorts',
  'video_date': '2026-05-11',
  'comment_date': '2026-05-11',
  'likes': 0,
  'collected_at': '2026-05-11'},
 {'id': 'yt_GncqjECzhOw_Ugz2U407CD5_9a3oyrh4AaABAg',
  'source_platform': 'youtube',
  'source_channel': 'VictorPonta1',
  'text_raw': 'Corect',
  'video_id': 'GncqjECzhOw',
  'video_title': 'Președintele Români

In [35]:
comments[0].keys()

dict_keys(['id', 'source_platform', 'source_channel', 'text_raw', 'video_id', 'video_title', 'video_date', 'comment_date', 'likes', 'collected_at'])

## 8. Curățare minimă a textului
Acum pornim de la `text_raw` și construim o variantă curățată în câmpul `text`.
Nu schimbăm sensul comentariului. Eliminăm doar zgomot simplu: linkuri, spații inutile, texte prea scurte și duplicate.

In [36]:
import re

def clean_text(text):
    text = re.sub(r"http\S+", "", text)      # elimină linkuri
    text = re.sub(r"\s+", " ", text)         # normalizează spațiile
    return text.strip()

## 9. Aplicăm curățarea
Pentru fiecare comentariu păstrăm textul original în `text_raw` și adăugăm textul curățat în `text`.

In [37]:
for comment in comments:
    comment["text"] = clean_text(comment["text_raw"])

comments[0]

{'id': 'yt_OCI4GZ8_XiU_UgwksPU95OgaL6b3SFp4AaABAg',
 'source_platform': 'youtube',
 'source_channel': 'VictorPonta1',
 'text_raw': 'Super tare domnul Ponta ❤❤❤',
 'video_id': 'OCI4GZ8_XiU',
 'video_title': 'Creator Digital vs. Românii: Banii din Dubai și Lecțiile #shorts',
 'video_date': '2026-05-11',
 'comment_date': '2026-05-11',
 'likes': 0,
 'collected_at': '2026-05-11',
 'text': 'Super tare domnul Ponta ❤❤❤'}

## 10. Filtrăm comentariile prea scurte
Pentru exercițiu păstrăm doar comentariile care au cel puțin 60 de caractere.
Comentariile foarte scurte sunt greu de interpretat în analiza discursivă.

In [38]:
MIN_CHARS = 60

comments_clean = [
    comment for comment in comments
    if len(comment["text"]) >= MIN_CHARS
]

print("Comentarii brute:", len(comments))
print("Comentarii după filtrarea lungimii:", len(comments_clean))

Comentarii brute: 48
Comentarii după filtrarea lungimii: 17


## 11. Filtrăm textele cu prea puține litere
Comentariile formate mai ales din emoji, simboluri sau caractere izolate produc zgomot.
Păstrăm comentariile în care cel puțin 50% dintre caractere sunt litere.

In [39]:
MIN_ALPHA = 0.5

def alpha_ratio(text):
    if len(text) == 0:
        return 0
    letters = sum(char.isalpha() for char in text)
    return letters / len(text)

comments_clean = [
    comment for comment in comments_clean
    if alpha_ratio(comment["text"]) >= MIN_ALPHA
]

print("Comentarii după filtrarea literelor:", len(comments_clean))

Comentarii după filtrarea literelor: 17


## 12. Eliminăm duplicatele
Dacă același text apare de mai multe ori, îl păstrăm o singură dată.

In [40]:
seen_texts = set()
unique_comments = []

for comment in comments_clean:
    text = comment["text"].lower()
    if text not in seen_texts:
        unique_comments.append(comment)
        seen_texts.add(text)

comments_clean = unique_comments

print("Comentarii finale după deduplicare:", len(comments_clean))

Comentarii finale după deduplicare: 17


## 14. Salvăm fișierul curățat
Salvăm rezultatul în `data/cleaned/`.

In [41]:
clean_output_file = ROOT / "data" / "cleaned" / f"{student_id}_youtube_clean.jsonl"
clean_output_file.parent.mkdir(parents=True, exist_ok=True)

with clean_output_file.open("w", encoding="utf-8") as f:
    for comment in comments_clean:
        f.write(json.dumps(comment, ensure_ascii=False) + "\n")

print("Comentarii curate salvate:", len(comments_clean))
print("Fișier:", clean_output_file)

Comentarii curate salvate: 17
Fișier: c:\Users\iarin\Desktop\inginerie AI\echochamber-project-team-6\data\cleaned\student_01_youtube_clean.jsonl


# Functia de curatare

In [42]:
import re

def clean_comments(comments, min_chars=60, min_alpha=0.5):
    cleaned = []
    seen_texts = set()
    
    for comment in comments:
        # 1. Curățare text
        text = comment["text_raw"]
        text = re.sub(r"http\S+", "", text)
        text = re.sub(r"\s+", " ", text).strip()
        
        # 2. Filtru lungime
        if len(text) < min_chars:
            continue
        
        # 3. Filtru proporție litere
        letters = sum(char.isalpha() for char in text)
        alpha_ratio = letters / len(text) if len(text) > 0 else 0
        
        if alpha_ratio < min_alpha:
            continue
        
        # 4. Filtru duplicate
        text_key = text.lower()
        if text_key in seen_texts:
            continue
        
        seen_texts.add(text_key)
        
        # 5. Păstrăm comentariul și adăugăm textul curățat
        new_comment = comment.copy()
        new_comment["text"] = text
        new_comment["lang"] = "ro"
        cleaned.append(new_comment)
    
    return cleaned

In [43]:
comments_clean = clean_comments(
    comments,
    min_chars=60,
    min_alpha=0.5
)

print("Comentarii brute:", len(comments))
print("Comentarii curate:", len(comments_clean))

Comentarii brute: 48
Comentarii curate: 17


In [44]:
for comment in comments_clean[:3]:
    print("RAW:", comment["text_raw"])
    print("CLEAN:", comment["text"])
    print("---")

RAW: HT.REZISTI! DE CE MI-ATI ȘTERS MESAJELE!? DE CE VĂ,,ESTE FRICĂ NU O SĂ SCĂPAȚI!" ( IISUS HRISTOS,,NU VĂ IARTĂ!") DACĂ  ÎL IUBIȚI PE ,,LUCIFER!"?) VĂ DUCE DIRECT LA ,,SMOALĂ!"? DRUM BUN! 😮😮😮😮😮😮😮😮
CLEAN: HT.REZISTI! DE CE MI-ATI ȘTERS MESAJELE!? DE CE VĂ,,ESTE FRICĂ NU O SĂ SCĂPAȚI!" ( IISUS HRISTOS,,NU VĂ IARTĂ!") DACĂ ÎL IUBIȚI PE ,,LUCIFER!"?) VĂ DUCE DIRECT LA ,,SMOALĂ!"? DRUM BUN! 😮😮😮😮😮😮😮😮
---
RAW: Poate aveti ocazia sa-i intrebati pe polonezi CUM au reusit sa ia 40 de miliarde si sa-i retina pe toti ptr. industria lor interna ? Any thoughts?
CLEAN: Poate aveti ocazia sa-i intrebati pe polonezi CUM au reusit sa ia 40 de miliarde si sa-i retina pe toti ptr. industria lor interna ? Any thoughts?
---
RAW: Mai de voi nu aveti contract la salubritate sa luati gunoiul din studio !
CLEAN: Mai de voi nu aveti contract la salubritate sa luati gunoiul din studio !
---


In [45]:
clean_output_file = ROOT / "data" / "cleaned" / f"{student_id}_youtube_clean.jsonl"
clean_output_file.parent.mkdir(parents=True, exist_ok=True)

with clean_output_file.open("w", encoding="utf-8") as f:
    for comment in comments_clean:
        f.write(json.dumps(comment, ensure_ascii=False) + "\n")

print("Fișier salvat:", clean_output_file)
print("Comentarii salvate:", len(comments_clean))

Fișier salvat: c:\Users\iarin\Desktop\inginerie AI\echochamber-project-team-6\data\cleaned\student_01_youtube_clean.jsonl
Comentarii salvate: 17


15. Ce am obținut
Am produs două fișiere:
- `data/raw/student_XX_youtube_raw.jsonl` — comentarii brute
- `data/cleaned/student_XX_youtube_clean.jsonl` — comentarii curățate
Fișierul curățat va putea fi unit cu fișierele celorlalți membri ai echipei.